In [1]:
## Load libraries
import numpy as np
import sympy as sp
import sys
import matplotlib.pyplot as plt
import matplotlib.cm as cm
plt.style.use('dark_background')
%matplotlib inline

In [2]:
## Mount the Google Drive folder, if needed, for accessing data
if('google.colab' in sys.modules):
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    # Change path below starting from /content/drive/MyDrive/Colab Notebooks/
    # depending on how data is organized inside your Colab Notebooks folder in
    # Google Drive
    DIR = '/content/drive/MyDrive/Colab Notebooks/MAHE/MSIS Coursework/OddSem2023MAHE'
    DATA_DIR = DIR+'/Data/'
else:
    DATA_DIR = 'Data/'

---

Calculating the softmax loss symbolically

Consider the following 3 images with 4 features each:

![](https://onedrive.live.com/embed?resid=37720F927B6DDC34%21101114&authkey=%21AEqz7RyEekVGRN8&width=1024)

---

In [3]:
# Define numpy arrays for each image
x1 = np.array([26, 100, 90, 80]) # Image-1
x2 = np.array([37, 58, 120, 96]) # Image-2
x3 = np.array([12, 30, 46, 54]) # Image-3

# Build symbolic data matrix
X = sp.Matrix(np.stack((x1, x2, x3 ), axis = 1))

# Pretty print symbolic matrix
sp.pprint(X)

⎡26   37   12⎤
⎢            ⎥
⎢100  58   30⎥
⎢            ⎥
⎢90   120  46⎥
⎢            ⎥
⎣80   96   54⎦


In [4]:
# Create symbolic weight matrix
W = sp.MatrixSymbol('W', 3, 4)

# Pretty print symbolic weight matrix
sp.pprint(sp.Matrix(W)) # Pretty print symbolic weight matrix

⎡W₀₀  W₀₁  W₀₂  W₀₃⎤
⎢                  ⎥
⎢W₁₀  W₁₁  W₁₂  W₁₃⎥
⎢                  ⎥
⎣W₂₀  W₂₁  W₂₂  W₂₃⎦


In [5]:
# Create symbolic bias vector
b = sp.MatrixSymbol('b', 3, 1)

# Pretty print symbolic weight matrix
sp.pprint(sp.Matrix(b)) # Pretty print symbolic weight matrix

⎡b₀₀⎤
⎢   ⎥
⎢b₁₀⎥
⎢   ⎥
⎣b₂₀⎦


In [6]:
# Calculate symbolic raw scores
z0 = W*X[:, 0] + b
z1 = W*X[:, 1] + b
z2 = W*X[:, 2] + b
sp.pprint(z0)

  ⎡26 ⎤    
  ⎢   ⎥    
  ⎢100⎥    
W⋅⎢   ⎥ + b
  ⎢90 ⎥    
  ⎢   ⎥    
  ⎣80 ⎦    


In [7]:
# Calculate symbolic softmax loss for each image
loss_1 = -sp.log(sp.exp(z0[0,0])/(sp.exp(z0[0,0])+sp.exp(z0[1,0])+sp.exp(z0[2,0])))
loss_2 = -sp.log(sp.exp(z1[1,0])/(sp.exp(z1[0,0])+sp.exp(z1[1,0])+sp.exp(z1[2,0])))
loss_3 = -sp.log(sp.exp(z2[2,0])/(sp.exp(z2[0,0])+sp.exp(z2[1,0])+sp.exp(z2[2,0])))

# Calculate symbolic average softmax loss
average_loss = (1/3)*(loss_1 + loss_2 + loss_3)

# Pretty print symbolic average softmax loss
sp.pprint(average_loss)

                       ⎛                                            12⋅W₂₀ + 3
                       ⎜                                           ℯ          
- 0.333333333333333⋅log⎜──────────────────────────────────────────────────────
                       ⎜ 12⋅W₀₀ + 30⋅W₀₁ + 46⋅W₀₂ + 54⋅W₀₃ + b₀₀    12⋅W₁₀ + 3
                       ⎝ℯ                                        + ℯ          

0⋅W₂₁ + 46⋅W₂₂ + 54⋅W₂₃ + b₂₀                                           ⎞     
                                                                        ⎟     
────────────────────────────────────────────────────────────────────────⎟ - 0.
0⋅W₁₁ + 46⋅W₁₂ + 54⋅W₁₃ + b₁₀    12⋅W₂₀ + 30⋅W₂₁ + 46⋅W₂₂ + 54⋅W₂₃ + b₂₀⎟     
                              + ℯ                                       ⎠     

                   ⎛                                             26⋅W₀₀ + 100⋅
                   ⎜                                            ℯ             
333333333333333⋅log⎜──────────────────────────────

In [8]:
# Evaluate average softmax loss for arbitrary weight and bias values
average_loss.evalf(subs = {W: sp.Matrix(np.random.randn(3,4)), b: sp.Matrix(np.random.randn(3,1))})

227.367575094558

---

Function to visualize gradient descent in 1D

---

In [9]:
## Credit: https://github.com/pablocpz/Gradient-Descent-Visualizations/blob/main/1D%20simulation.ipynb
## Function for visualizing gradient descent in 1D
from matplotlib import animation

def grad_1D(expr, w_values, w_initial, learning_rate, training_epochs, display_animation=False):
    """
    expr : Sympy expression like (w+2)**2+3 (w symbol)
    w_values : np.linspace(a, b, n)
    w_initial: starting value
    learning_rate, training_epochs = z, r
    display_animation : if True, will return two objects to visualizing it (example below)
    """

    w = sp.symbols("w")
    func = sp.lambdify(w, expr, "numpy")
    deriv = sp.diff(expr)
    deriv_func = sp.lambdify(w, deriv, "numpy")

    #algorithm
    local_min = w_initial#np.random.choice(w_values,1)
    initial_local_min = local_min
    print(f"Initial loss value {initial_local_min}")
    model_params = np.zeros((training_epochs, 2)) #shape epochs x 2 cols
    for i in range(0, training_epochs):
        grad = deriv_func(local_min)
        local_min = local_min - (grad*learning_rate)
        model_params[i,0] = local_min
        model_params[i,1] = grad
    print(f"Final loss value {training_epochs} epochs: {local_min}")


    if display_animation:
        #prepare animation
        grad_fig, ax = plt.subplots(figsize=(6, 4), dpi=100)

        ax.plot(w_values, func(w_values), label=f"${sp.latex(expr)}$")
        #ax.plot(w_values, deriv_func(w_values), label=f"dL/dw ${sp.latex(deriv)}$")

        plt.title(f"Empirical Local Minimum: {local_min}")


        plt.axhline(0, color='white',linewidth=0.5)
        plt.axvline(0, color='white',linewidth=0.5)

        plt.grid(color="gray", linestyle="--", linewidth=0.5)
        plt.xlabel("w");
        plt.ylabel("L(w)")
        #plt.legend();
        plt.close()

        def tangent_line(x, x1, y1):
            # m*x+b
            return deriv_func(x1)*(x-x1) + y1

        title = ax.set_title('', fontweight="bold")
        local_min_scat = ax.scatter(initial_local_min, func(initial_local_min), color="orange")
        initial_tangent_range = np.linspace(initial_local_min-0.5, initial_local_min+0.5, 10)
        tangent_plot = ax.plot(initial_tangent_range, tangent_line(x=initial_tangent_range,
                                                                   x1=initial_local_min,
                                                                   y1=func(initial_local_min)), linestyle="--",  color="orange", linewidth=2)[0]
        grad_annotation = ax.annotate(
            'Gradient={0:2f}'.format(deriv_func(initial_local_min)),
            xy=(initial_local_min,func(initial_local_min)), xytext=(initial_local_min,func(initial_local_min)+1),
            arrowprops = {'arrowstyle': "-", 'facecolor' : 'orange'},
            textcoords='data', color='orange' , rotation=20, fontweight="bold"
        )

        def drawframe(epoch):
            title.set_text('Epoch={0:4d}, learning rate = {1:2g}'.format(epoch,  learning_rate))
            x1 = model_params[epoch, 0]
            y1 = func(model_params[epoch, 0])
            local_min_scat.set_offsets((x1, y1))
            tangent_range = np.linspace(x1-0.5, x1+0.5, 10)
            tangent_values = tangent_line(x=tangent_range, x1=x1 ,y1=y1)
            tangent_plot.set_xdata(tangent_range)
            tangent_plot.set_ydata(tangent_values)
            grad_annotation.set_position((x1, y1+1))
            grad_annotation.xy = (x1, y1)
            grad_annotation.set_text('Gradient={0:2f}'.format(model_params[epoch, 1]))
            return local_min_scat,

        # blit=True re-draws only the parts that have changed.

        anim = animation.FuncAnimation(grad_fig, drawframe, frames=training_epochs, repeat=False, interval=500, blit=True)


        writer = animation.PillowWriter(fps=30,
                                        metadata=dict(artist='Me'),
                                        bitrate=1800)
        # ani.save('gradient1D.gif', writer=writer)
        # from IPython.display import HTML
        # HTML(anim.to_html5_video())

        return anim, writer

---

Gradient descent visualization in 1D applied to $L(w) = (w+2)^2+3$ starting from $w=2.$

---

In [10]:
## Generate animation for gradient descent in 1D
w = sp.symbols('w')
anim, writer = grad_1D(expr=(w+2)**2+3,
                       w_values=np.linspace(-6,6, 20),
                       w_initial = 2.0, learning_rate=0.1,
                       training_epochs=100,
                       display_animation=True)
#anim.save(DATA_DIR+'gradient1D.gif', writer=writer) #if you want to save it as GIF
# from IPython.display import HTML
# HTML(anim.to_html5_video()) #render the video

Initial loss value 2.0
Final loss value 100 epochs: -1.9999999991851856


---

Gradient descent in 1D applied to $L(w) = (w+2)^2+3$ starting from $w=2.$

----

In [11]:
## Gradient descent in 1D
L = lambda w: (w+2)**2 + 3
gradL = lambda w: 2*(w+2)
# Try 1e-05 (slow learning rate), 1e-01 (optimal),
# 1e0 (oscillates and does not converge),
# and 0.95 (oscillates towards the end and converges)
alpha = 0.95 # learning rate
tol = 1e-05 # stopping tolerance
iter = 0
maxiter = 1000

w = 2 # starting point

# Learning process
while np.abs(gradL(w)) > tol and iter < maxiter:
  w = w + alpha * -gradL(w)
  iter = iter+1
  print('Iteration = %d, w = %f, gradL(w) = %f'%(iter, w, gradL(w)))

Iteration = 1, w = -5.600000, gradL(w) = -7.200000
Iteration = 2, w = 1.240000, gradL(w) = 6.480000
Iteration = 3, w = -4.916000, gradL(w) = -5.832000
Iteration = 4, w = 0.624400, gradL(w) = 5.248800
Iteration = 5, w = -4.361960, gradL(w) = -4.723920
Iteration = 6, w = 0.125764, gradL(w) = 4.251528
Iteration = 7, w = -3.913188, gradL(w) = -3.826375
Iteration = 8, w = -0.278131, gradL(w) = 3.443738
Iteration = 9, w = -3.549682, gradL(w) = -3.099364
Iteration = 10, w = -0.605286, gradL(w) = 2.789428
Iteration = 11, w = -3.255242, gradL(w) = -2.510485
Iteration = 12, w = -0.870282, gradL(w) = 2.259436
Iteration = 13, w = -3.016746, gradL(w) = -2.033493
Iteration = 14, w = -1.084928, gradL(w) = 1.830143
Iteration = 15, w = -2.823565, gradL(w) = -1.647129
Iteration = 16, w = -1.258792, gradL(w) = 1.482416
Iteration = 17, w = -2.667087, gradL(w) = -1.334175
Iteration = 18, w = -1.399621, gradL(w) = 1.200757
Iteration = 19, w = -2.540341, gradL(w) = -1.080681
Iteration = 20, w = -1.513693, gr

---

Gradient Descent in 2D applied to $L(\mathbf{w}) = (w_1-2)^2+(w_2+3)^2$ starting from $w_1=1, w_2=1.$

---

In [12]:
# Gradient descent in 2D
L = lambda w: (w[0]-2)**2 + (w[1]+3)**2
gradL = lambda w: np.array([2*(w[0]-2), 2*(w[1]+3)])
alpha = 1e-02 # learning rate
tol = 1e-05 # stopping tolerance
iter = 0
maxiter = 1000

w =  np.array([1, 1]) # initial guess

while np.linalg.norm(gradL(w)) > tol and iter < maxiter:
  w = w + alpha *(-gradL(w))
  iter = iter+1
  print('Iteration = %d, w1 = %f, w2 = %f, ||gradL(w)|| = %f'%(iter, w[0], w[1], np.linalg.norm(gradL(w))))

Iteration = 1, w1 = 1.020000, w2 = 0.920000, ||gradL(w)|| = 8.081287
Iteration = 2, w1 = 1.039600, w2 = 0.841600, ||gradL(w)|| = 7.919661
Iteration = 3, w1 = 1.058808, w2 = 0.764768, ||gradL(w)|| = 7.761268
Iteration = 4, w1 = 1.077632, w2 = 0.689473, ||gradL(w)|| = 7.606043
Iteration = 5, w1 = 1.096079, w2 = 0.615683, ||gradL(w)|| = 7.453922
Iteration = 6, w1 = 1.114158, w2 = 0.543370, ||gradL(w)|| = 7.304843
Iteration = 7, w1 = 1.131874, w2 = 0.472502, ||gradL(w)|| = 7.158747
Iteration = 8, w1 = 1.149237, w2 = 0.403052, ||gradL(w)|| = 7.015572
Iteration = 9, w1 = 1.166252, w2 = 0.334991, ||gradL(w)|| = 6.875260
Iteration = 10, w1 = 1.182927, w2 = 0.268291, ||gradL(w)|| = 6.737755
Iteration = 11, w1 = 1.199269, w2 = 0.202925, ||gradL(w)|| = 6.603000
Iteration = 12, w1 = 1.215283, w2 = 0.138867, ||gradL(w)|| = 6.470940
Iteration = 13, w1 = 1.230978, w2 = 0.076090, ||gradL(w)|| = 6.341521
Iteration = 14, w1 = 1.246358, w2 = 0.014568, ||gradL(w)|| = 6.214691
Iteration = 15, w1 = 1.261431

---

Calculating softmax loss and gradient for a toy dataset

----

In [34]:
# Generate artificial data with 5 samples, 4 features per sample
# and 3 output classes
num_samples = 5 # number of samples
num_features = 4 # number of features (a.k.a. dimensionality)
num_labels = 3 # number of output labels
# Data matrix (each column = single sample)
X = np.random.choice(np.arange(0, 5), size = (num_features, num_samples), replace = True)
# Class labels
y = np.random.choice([0, 1, 2], size = num_samples, replace = True)
# Randomly assign entries of weights matrix
W = np.random.choice(np.arange(-4, 4), size = (num_labels, num_features), replace = True)
print('X = ')
print(X)
print('y = ')
print(y)
print('W = ')
print(W)

X = 
[[1 2 1 0 1]
 [2 1 2 2 0]
 [2 0 0 1 4]
 [1 0 0 4 1]]
y = 
[2 2 0 1 2]
W = 
[[-4  3 -1 -3]
 [-4  2 -3  0]
 [-1  1 -3  3]]


In [35]:
# Add the bias feature to the data matrix (run this cell only once!)
print('X = ')
print(X)
print('X with bias feature = ')
X = np.vstack([X, np.ones((1, num_samples))])
print(X)

X = 
[[1 2 1 0 1]
 [2 1 2 2 0]
 [2 0 0 1 4]
 [1 0 0 4 1]]
X with bias feature = 
[[1. 2. 1. 0. 1.]
 [2. 1. 2. 2. 0.]
 [2. 0. 0. 1. 4.]
 [1. 0. 0. 4. 1.]
 [1. 1. 1. 1. 1.]]


In [36]:
# Adjust the weight matrix with (possibly random) values added
# for bias as the last column (run this cell only once!)
W = np.hstack([W, np.ones((num_labels, 1))])
print(W)

[[-4.  3. -1. -3.  1.]
 [-4.  2. -3.  0.  1.]
 [-1.  1. -3.  3.  1.]]


In [16]:
# Calculate the raw zcores matrix
Z =  np.dot(W, X)
print('Z = ')
print(Z)

Z = 
[[  1.  -8.  -7. -16.   1.]
 [ -1.   7.   7.  15.   0.]
 [  1.  -6.  -2.  -6.   1.]]


In [78]:
#define softmax function
def softmax(Z):
    # Convert scores to non-normalized probabilites matrix. Note that for each sample,
    # that is in each column, the values don't add up to 1. Also note that the
    # output values are typically large or small
    Z_exp = np.exp(Z - np.max(Z, axis=0))
    # Normalize probabilities matrix such that the sum across each column is equal to 1.
    # Now we have actually probability values for each sample.
    return Z_exp / np.sum(Z_exp, axis=0)

In [80]:
# Calculate the probabilities matrix
# P = softmax(Z_exp)
P = P / np.sum(P, axis = 0)
print('P = ', P)
print(np.sum(P, axis = 0))   
# Print the correct label for each sample
print(y)

P =  [[-6.93764562e-12 -4.67881141e-14  1.50000000e+00 -1.08926068e-03
  -8.99270406e-03]
 [ 1.49954447e+00 -7.61498976e-09 -3.34642546e-03  1.44052838e+00
   1.49997771e+00]
 [-4.99544474e-01  1.00000001e+00 -4.96653574e-01 -4.39439121e-01
  -4.90985005e-01]]
[1. 1. 1. 1. 1.]
[1 2 0 1 1]


In [81]:
# Calculate loss for all samples.
loss =  -np.log(P[y, np.arange(num_samples)])
print('Loss = ')
print(loss)
# Calculate average training loss
loss_data = np.mean(loss)
print('Total loss = %f'%(loss_data))

Loss = 
[-4.05161378e-01 -7.61503657e-09 -4.05465108e-01 -3.65009978e-01
 -4.05450248e-01]
Total loss = -0.316217


In [82]:
# Adjust the probability matrix such that 1 is subtracted
# from each samples correct category probability
P[y, range(num_samples)] = P[y, range(num_samples)] - 1

In [83]:
# Adjust the probability matrix such that 1 is subtracted
# from each samples correct category probability
# Calculate the gradient of total loss w.r.t. the weights W
P[y, range(num_samples)] = P[y, range(num_samples)] - 1 
dW_data = (1/num_samples)*np.dot(P, X.T)
print(dW_data)

[[-0.10359708 -0.40087141 -0.10762987 -0.4004357  -0.10201639]
 [-0.40086042 -0.45025444 -0.72456687 -0.52673911 -0.31265917]
 [-1.2955425  -0.94887416 -1.36780326 -1.47282519 -0.58532443]]


---

Applying $L_2$-regularization (loss and gradient)

---

In [84]:
#regularization loss
alpha = 0.01 # regularization strength = 10%
loss_reg = np.sum(W[:, :-1] * W[:, :-1])
print("Total loss = %f"%(loss_data + alpha*loss_reg))

Total loss = 0.693783


In [87]:
# Calculate the gradient of total loss w.r.t. the weights W
dW = (1/num_samples)*np.dot(P, X.T) + alpha * 2 * np.hstack([W[:, :-1], np.zeros((num_labels, 1))])
print(dW)

[[-0.16359708 -0.44087141 -0.04762987 -0.4804357  -0.10201639]
 [-0.48086042 -0.53025444 -0.68456687 -0.46673911 -0.31265917]
 [-1.2755425  -1.00887416 -1.32780326 -1.43282519 -0.58532443]]


In [91]:
# Apply gradient descent to the toy dataset
alpha = 1e-02 # learning rate
tol = 1e-05 # stopping tolerance
iter = 0
maxiter = 1000

w = 2 # starting point

# Learning process
while np.linalg.norm(dW) > tol and iter < maxiter:
  w = w + alpha * (-dW)
  iter = iter+1
  print('Iteration = %d, ||dW|| = %f'%(iter, np.linalg.norm(dW)))

Iteration = 1, ||dW|| = 2.926559
Iteration = 2, ||dW|| = 2.926559
Iteration = 3, ||dW|| = 2.926559
Iteration = 4, ||dW|| = 2.926559
Iteration = 5, ||dW|| = 2.926559
Iteration = 6, ||dW|| = 2.926559
Iteration = 7, ||dW|| = 2.926559
Iteration = 8, ||dW|| = 2.926559
Iteration = 9, ||dW|| = 2.926559
Iteration = 10, ||dW|| = 2.926559
Iteration = 11, ||dW|| = 2.926559
Iteration = 12, ||dW|| = 2.926559
Iteration = 13, ||dW|| = 2.926559
Iteration = 14, ||dW|| = 2.926559
Iteration = 15, ||dW|| = 2.926559
Iteration = 16, ||dW|| = 2.926559
Iteration = 17, ||dW|| = 2.926559
Iteration = 18, ||dW|| = 2.926559
Iteration = 19, ||dW|| = 2.926559
Iteration = 20, ||dW|| = 2.926559
Iteration = 21, ||dW|| = 2.926559
Iteration = 22, ||dW|| = 2.926559
Iteration = 23, ||dW|| = 2.926559
Iteration = 24, ||dW|| = 2.926559
Iteration = 25, ||dW|| = 2.926559
Iteration = 26, ||dW|| = 2.926559
Iteration = 27, ||dW|| = 2.926559
Iteration = 28, ||dW|| = 2.926559
Iteration = 29, ||dW|| = 2.926559
Iteration = 30, ||dW|| 

---

Batch Processing

---

In [26]:
# Demonstration of splitting samples into batches for batch processing
# using a simple example

num_samples = 11 # total number of samples
num_iters = 10   # number of iterations
batch_size = 3   # number of samples for calculating loss and gradient in each iteration

print('Number of samples = %d'%(num_samples))
print('Number of iterations = %d'%(num_iters))
print('Batch size = %d'%(batch_size))

# Function to generate sample indices for batch processing according to batch size
def generate_batch_indices():
  # Reorder sample indices
  reordered_sample_indices = np.random.choice(num_samples, num_samples, replace = False)
  # Generate batch indices for batch processing
  batch_indices = np.split(reordered_sample_indices, np.arange(batch_size, len(reordered_sample_indices), batch_size))
  return(batch_indices)

# Number of batches per epoch
num_iterations_per_epoch = int(np.ceil(num_samples/batch_size))
print('Number of iterations per epoch = %d\n'%(num_iterations_per_epoch))
b = 0
epoch = 0
for it in range(num_iters):
  if it % num_iterations_per_epoch == 0:# check if we are at the start of an epoch
    print('--------------------------------')
    print('Epoch %d:'%(epoch+1))
    batch_indices = generate_batch_indices()
    b = 0
    epoch = epoch + 1
    print('--------------------------------')
  print('In iteration %d, using samples' % (it+1))
  print(batch_indices[b])
  b += 1

Number of samples = 11
Number of iterations = 10
Batch size = 3
Number of iterations per epoch = 4

--------------------------------
Epoch 1:
--------------------------------
In iteration 1, using samples
[2 6 7]
In iteration 2, using samples
[9 3 5]
In iteration 3, using samples
[ 8  1 10]
In iteration 4, using samples
[0 4]
--------------------------------
Epoch 2:
--------------------------------
In iteration 5, using samples
[9 3 7]
In iteration 6, using samples
[0 6 1]
In iteration 7, using samples
[5 8 4]
In iteration 8, using samples
[ 2 10]
--------------------------------
Epoch 3:
--------------------------------
In iteration 9, using samples
[1 2 0]
In iteration 10, using samples
[9 8 5]
